<a href="https://colab.research.google.com/github/debayandeb575-svg/first-proj/blob/main/debayan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:


from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
import os
dataset_path = "/content/drive/MyDrive/placement_data.csv"
# os.listdir is used to list contents of a directory.
# If you intend to read the file, you would typically use a function like pandas.read_csv()
# or open() for general file reading.
# If you just want to check if the file exists, you can use os.path.exists()
print(f"File exists: {os.path.exists(dataset_path)}")
print(f"Path is a file: {os.path.isfile(dataset_path)}")
# If you want to list the directory containing the file, use:
# print(os.listdir(os.path.dirname(dataset_path)))

File exists: False
Path is a file: False


In [9]:
import pandas as pd
df = pd.read_csv(dataset_path)
df['dsa_cgpa_ratio']=df['DSA_Solved']/df['CGPA']

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

import pandas as pd # Ensure pandas is imported if not already in scope

# Re-read the CSV and re-calculate dsa_cgpa_ratio to ensure a fresh df
df = pd.read_csv(dataset_path)
df['dsa_cgpa_ratio'] = df['DSA_Solved'] / df['CGPA']

# 1. Encode target + categorical columns
le = LabelEncoder()
df['Placement'] = le.fit_transform(df['Placement']) # Yes=1, No=0
df['Internship'] = le.fit_transform(df['Internship']) # Yes=1, No=0

# One-hot encode Branch
df = pd.get_dummies(df, columns=['Branch'], drop_first=True)

# 2. Split features + target
X = df.drop('Placement', axis=1)
y = df['Placement']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train 3 models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred))
    print("-"*40)

# 4. Pick best model
best_model_name = max(results, key=results.get)
print(f"\nBest Model: {best_model_name} with {results[best_model_name]*100:.2f}% accuracy")

Logistic Regression Accuracy: 95.00%
              precision    recall  f1-score   support

           0       0.97      0.89      0.93        38
           1       0.94      0.98      0.96        62

    accuracy                           0.95       100
   macro avg       0.95      0.94      0.95       100
weighted avg       0.95      0.95      0.95       100

----------------------------------------
Random Forest Accuracy: 89.00%
              precision    recall  f1-score   support

           0       0.85      0.87      0.86        38
           1       0.92      0.90      0.91        62

    accuracy                           0.89       100
   macro avg       0.88      0.89      0.88       100
weighted avg       0.89      0.89      0.89       100

----------------------------------------
XGBoost Accuracy: 91.00%
              precision    recall  f1-score   support

           0       0.85      0.92      0.89        38
           1       0.95      0.90      0.93        62

    acc

In [11]:
rf = models["Random Forest"]
importances = rf.feature_importances_
pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values('Importance', ascending=False)

,Feature,Importance
0,CGPA,0.503816
1,DSA_Solved,0.202055
4,dsa_cgpa_ratio,0.130258
2,Projects,0.097429
3,Internship,0.032424
8,Branch_ME,0.013615
6,Branch_EE,0.009397
7,Branch_IT,0.005730
5,Branch_ECE,0.005274


In [12]:
import joblib
joblib.dump(models[best_model_name], 'placement_model.pkl')

['placement_model.pkl']

In [5]:
import subprocess
subprocess.run(["pip", "install", "pyngrok", "-q"])
from pyngrok import ngrok

# NOTE: You should only run this cell once per session or if your token changes.
# If you share this notebook, be careful not to expose your token.
ngrok.set_auth_token("3ECpyxXLrvbDPP3aXHdh13gS6p7_5nFqpVyWpMCPgYE6gkYVr")

In [24]:
%%writefile app.py
import streamlit as st

import pandas as pd
import joblib

model = joblib.load('placement_model.pkl')
REQUIRED_COLS = list(model.feature_names_in_) # Get exact order from model

st.title("🎓 Placement Predictor")

cgpa = st.slider("CGPA", 6.0, 10.0, 8.0, 0.1)
dsa = st.slider("DSA Problems Solved", 0, 300, 150)
projects = st.slider("Projects", 0, 10, 3)
internship = 1 if st.selectbox("Internship", ["No", "Yes"]) == "Yes" else 0
branch = st.selectbox("Branch", ["CSE", "IT", "ECE", "ME", "CIVIL", "EE"])

dsa_cgpa_ratio = dsa / cgpa

# Build dict with all possible columns
input_dict = {
    'CGPA': cgpa,
    'DSA_Solved': dsa,
    'Projects': projects,
    'Internship': internship,
    'dsa_cgpa_ratio': dsa_cgpa_ratio,
    'Branch_CIVIL': 1 if branch == 'CIVIL' else 0,
    'Branch_ECE': 1 if branch == 'ECE' else 0,
    'Branch_EE': 1 if branch == 'EE' else 0,
    'Branch_IT': 1 if branch == 'IT' else 0,
    'Branch_ME': 1 if branch == 'ME' else 0,
}

# CRITICAL FIX: Create dataframe using exact order from model
input_data = pd.DataFrame([input_dict])[REQUIRED_COLS]

if st.button("Predict"):
    prob = model.predict_proba(input_data)[0][1] * 100
    st.metric("Placement Probability", f"{prob:.1f}%")

    # WOW factor sliders
    new_dsa = st.slider("What if DSA=?", 0, 500, dsa, key="wow")
    new_ratio = new_dsa / cgpa
    input_dict['DSA_Solved'] = new_dsa
    input_dict['dsa_cgpa_ratio'] = new_ratio
    new_input = pd.DataFrame([input_dict])[REQUIRED_COLS]
    new_prob = model.predict_proba(new_input)[0][1] * 100
    st.info(f"New chance: {new_prob:.1f}% | Delta: +{new_prob-prob:.1f}%")

Overwriting app.py


In [18]:
# This cell's content has been moved into the app.py definition in cell `cuwX2204wWzN` to resolve the NameError.

In [19]:
!pip install pyngrok -q
from pyngrok import ngrok
import subprocess
import os
import sys # Import sys
import time

# Ensure streamlit is installed for the subprocess environment
!pip install streamlit -q

# Terminate any previous ngrok tunnels to avoid conflicts
ngrok.kill()

# Check if app.py exists before attempting to launch Streamlit
app_file_path = '/content/app.py'
if not os.path.exists(app_file_path):
    print(f"Error: The app.py file was not found at {app_file_path}.\nPlease ensure cell `1fc0fda7` (%%writefile app.py) was executed successfully.")
else:
    print(f"Found app.py at {app_file_path}. Proceeding to launch Streamlit.")
    # Run Streamlit in the background
    streamlit_process = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', app_file_path], # Use absolute path
        stdout=subprocess.PIPE, # Capture stdout
        stderr=subprocess.PIPE, # Capture stderr
        preexec_fn=os.setsid
    )

    # Give Streamlit a moment to start up
    time.sleep(5)

    try:
        # Open a ngrok tunnel to the Streamlit port (default is 8501)
        public_url = ngrok.connect(addr='8501', proto='http') # Corrected ngrok.connect call
        print(f"Streamlit App URL: {public_url}")
        print("To stop the Streamlit app and ngrok tunnel, restart the Colab runtime or run `ngrok.kill()`.")

    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Attempting to read Streamlit logs...")
        # Use communicate with a timeout to prevent hanging if process doesn't terminate
        try:
            stdout, stderr = streamlit_process.communicate(timeout=10)
            if stdout:
                print("Streamlit stdout:")
                print(stdout.decode('utf-8'))
            if stderr:
                print("Streamlit stderr:")
                print(stderr.decode('utf-8'))
        except subprocess.TimeoutExpired:
            print("Streamlit process did not terminate within timeout to read logs. Killing it...")
            streamlit_process.kill() # Terminate the process if it timed out
            stdout, stderr = streamlit_process.communicate() # Read any remaining buffered output
            if stdout: print("Streamlit stdout after kill:", stdout.decode('utf-8'))
            if stderr: print("Streamlit stderr after kill:", stderr.decode('utf-8'))
        print("Please check your ngrok authtoken in cell `724f7770` and ensure 'app.py' is correctly written and accessible.")

    # Keep the Streamlit process running in the background.
    # It will be killed if the Colab runtime restarts or if `ngrok.kill()` is explicitly called.
    # You might want to store `streamlit_process` if you need to manually terminate it later.

Found app.py at /content/app.py. Proceeding to launch Streamlit.
Streamlit App URL: NgrokTunnel: "https://steadying-renewal-tactful.ngrok-free.dev" -> "http://localhost:8501"
To stop the Streamlit app and ngrok tunnel, restart the Colab runtime or run `ngrok.kill()`.


In [27]:
from google.colab import files
files.download('placement_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>